# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'professional network',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
nvidia/LocateAnything-3B
Updated
8 days ago
•
91.8k
•
1.22k
LiquidAI/LFM2.5-8B-A1B
Updated
about 2 hours ago
•
72.1k
•
487
HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
Apr 17
•
2.65M
•
1.37k
google/gemma-4-12B-it
Updated
about 13 hours ago
•
14.9k
•
298
openbmb/MiniCPM5-1B
U

In [20]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/LocateAnything-3B\nUpdated\n8 days ago\n•\n91.8k\n•\n1.22k\nLiquidAI/LFM2.5-8B-A1B\nUpdated\nabout 2 hours ago\n•\n72.1

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 19 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the premier collaboration platform for the machine learning (ML) community. It acts as a central hub where anyone—from researchers and engineers to AI enthusiasts—can share, explore, discover, and experiment with open-source ML models, datasets, and applications. Focused on empowering the next generation of ML engineers, scientists, and end-users, Hugging Face fosters an open, ethical AI future by enabling collaboration and innovation on a global scale.

The Hugging Face Hub hosts over **2 million models** and **500,000 datasets**, supporting cutting-edge research and real-world AI applications. Its extensive ecosystem includes spaces for running ML applications, enterprise-grade inference services, and tools to accelerate ML model deployment.

---

## What We Offer

- **Model Hub**: Browse, download, and fine-tune hundreds of thousands of pretrained models covering NLP, computer vision, speech, and reinforcement learning tasks.
- **Datasets**: Access a massive repository of public datasets for training and benchmarking ML systems.
- **Spaces**: Build and deploy AI applications for public use within the community.
- **Enterprise Solutions**: Scalable, secure support for organizations, including inference endpoints, storage buckets, and professional services.
- **Community Tools**: Forums, Discord channels, blogs, and daily research papers keep the community engaged and informed.
- **HuggingChat**: An AI-powered chat tool leveraging community models for interactive experiences.
- **Hugging Face PRO**: Tailored advanced features for teams and enterprises tackling complex ML challenges.

---

## Company Culture & Values

Hugging Face prides itself on being a **community-first, open-source, and ethical AI company.** Collaboration and transparency lie at the core of its culture, creating an environment where innovative ideas flourish and knowledge is shared freely. The company actively promotes diversity and inclusion within the AI landscape and values the impact of collective intelligence over isolated innovation.

Employees and contributors are driven by a passion for:

- **Advancing AI responsibly** by fostering ethical practices and openness.
- **Sharing knowledge and tools** with the global ML community.
- **Encouraging creativity and experimentation** to push the boundaries of technology.
- **Building long-term relationships** with users, contributors, and organizations.

---

## Customers & Community

Hugging Face serves a diverse customer base including:

- **Researchers and Data Scientists**: Accelerating research with easy access to prebuilt resources.
- **Startups and Developers**: Prototyping and deploying AI applications rapidly.
- **Large Enterprises**: Leveraging scalable infrastructure and enterprise support for production-grade AI solutions.
- **Academia**: Facilitating education and collaboration with open data and models.
- **AI Enthusiasts**: Empowering learning and experimentation.

The community is vibrant, with millions of members engaging daily in forums, open-source projects, and knowledge sharing on platforms like GitHub and Discord.

---

## Careers

Hugging Face is continually growing and offers exciting career opportunities for those passionate about AI and open-source technology. The company hires talents in a variety of roles including:

- Machine Learning Engineering
- Research Science
- Software Development
- Product Management
- Community and Developer Relations
- Enterprise and Customer Success

Working at Hugging Face means joining a mission-driven team dedicated to shaping the AI revolution, supported by inclusive culture and cutting-edge technologies.

Explore current openings and find how you can contribute at: [Hugging Face Careers](https://huggingface.co/careers)

---

## Get Involved

- Join the community on [Discord](https://discord.gg/huggingface) and [Forum](https://discuss.huggingface.co/)
- Contribute to open-source projects on [GitHub](https://github.com/huggingface)
- Stay updated via the [Hugging Face Blog](https://huggingface.co/blog)
- Download official brand assets and explore the [Hugging Face brand universe](https://huggingface.co/brand-assets)

---

## Contact & Connect

Website: https://huggingface.co  
Twitter: [@huggingface](https://twitter.com/huggingface)  
LinkedIn: https://linkedin.com/company/hugging-face

---

### Join Hugging Face – Together, let's build the future of AI.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

## Who We Are  
Hugging Face is the AI community building the future of machine learning. We are the collaboration platform where the machine learning community creates, shares, and improves models, datasets, and applications. Our mission is to empower developers, researchers, and enterprises globally to innovate faster and more effectively in AI.

---

## What We Offer  
- **Hugging Face Hub:** The central platform hosting over 2 million models and 500,000+ datasets, allowing seamless discovery, use, and collaboration on machine learning assets.  
- **Spaces:** Collaborative environments where ML applications run and interact in real-time, with thousands of featured projects ranging from image generation to video avatars.  
- **Buckets & Storage:** Scalable storage solutions tailored for AI models and datasets supporting enterprise needs.  
- **Enterprise Solutions:** Dedicated support, inference endpoints, and provider integrations designed for business-grade AI deployment.  
- **HuggingChat:** Conversational AI tools accelerating chatbot development and natural language applications.

---

## Our Community  
Hugging Face thrives as a vibrant, global community comprising researchers, data scientists, developers, and organizations. Connect with us through multiple channels such as:  
- **Discord & Forum** for direct interaction and support  
- **Blog & Daily Papers** to stay ahead with AI research insights  
- **GitHub** for collaborative open-source contributions  
- **Events & Learning resources** to enhance skills and network  

Our platform facilitates unlimited public hosting and collaboration, empowering both individual contributors and large teams to build and deploy cutting-edge AI faster.

---

## Company Culture  
At Hugging Face, we value:  
- **Openness & Collaboration** – We foster an inclusive community where sharing knowledge fuels innovation.  
- **Innovation & Impact** – We build tools that accelerate AI development and bring transformative technology to all.  
- **Transparency** – We maintain clear communication with our users and partners to build trust.  
- **User-first Mindset** – We design every product to empower our diverse users from hobbyists to enterprises.

---

## Careers & Opportunities  
Join a passionate, fast-growing team shaping the AI landscape. We offer roles across engineering, research, product, and community teams focused on:  
- Scaling AI infrastructure and tooling  
- Advancing natural language processing, computer vision, and multimodal AI  
- Building accessible AI applications and ecosystems  

At Hugging Face, you will collaborate with world-class AI experts, contribute to open-source projects, and work in a flexible, inclusive environment that supports professional growth.

---

## Our Customers & Partners  
We serve a broad spectrum of users including:  
- **AI researchers and academic institutions** accelerating model development and sharing  
- **Technology companies and startups** integrating AI into products and services  
- **Enterprises across industries** leveraging AI for innovation, supported by our enterprise-grade tools and dedicated support  

---

## Get Started  
Explore the future of AI with us today:  
- Browse and deploy models from over 2 million options  
- Collaborate on cutting-edge datasets and applications  
- Join our community on Discord and the Forum  
- Discover enterprise solutions tailored to your needs  

Visit [huggingface.co](https://huggingface.co) to dive in and be part of the AI revolution.

---

## Brand Assets  
Our official branding including logos and color palettes is available for your projects. We promote consistent, vibrant identity to reflect the energy and openness of our community.

**Colors:**  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280

---

Hugging Face — The AI community building the future. Join us and transform what’s possible in machine learning.

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Welcome to Hugging Face: Where AI Hugs the Future

---

## Who Are We?

Hugging Face is not just a company — it’s a vibrant AI community building the future, one model at a time. We're basically the cool hangout spot where machine learning enthusiasts, researchers, and developers gather to share, innovate, and collaborate on cutting-edge AI models, datasets, and applications.

---

## Our Playground: The Hugging Face Hub

- **Models Galore:** Over **2 million models** ready to be explored, from NVIDIA’s LocateAnything (superpower: object detection) to emergency creativity with LongCat-Video-Avatar (talking-head video magic!).  
- **Datasets Wonderland:** Browse over **500,000 datasets** spanning everything from ultra-fine web content to structured Wikipedia wonders.  
- **Spaces & Apps:** Host or glide through stunning AI-powered apps and cutting-edge experiments, like generating videos from images or 3D models conjured from a single snap.  

Think of it as the ultimate AI playground — with unlimited hosting, collaboration, and open-source good vibes.  

---

## Culture: Hugging Face Spirit 🤗

- **Open Collaboration:** We believe in collective brainpower. Share your art (a.k.a models), borrow others’, remix, and build better AI together.  
- **Community-Driven:** Whether you're a first-time coder or a seasoned AI guru, our **Discord, Forum, and GitHub** channels keep the party going 24/7.  
- **Friendly AI Geeks & Innovators:** No suits, no jargon-filled boardrooms—just passionate folks who want to help machines learn faster and friendlier.  
- **Brand Colors:** Rocking the vibrant yellow (#FFD21E), fiery orange (#FF9D00), and classic cool gray (#6B7280)—because we shine bright and keep it grounded!  

---

## Customers & Collaborators  

From the bustling tech startups hunting for AI superpowers to multinational enterprises looking to turbocharge their machine learning workflows — everyone’s invited to the Hugging Face fiesta.

- **Enterprises:** Get premium support, inference endpoints, and pro-tier tools to take AI projects from "it works?" to "it wows!"  
- **Researchers & Developers:** Tap into trending models like LiquidAI’s latest LFM or Google’s multitask marvels.  
- **HuggingChat:** Fancy talking AI? Connect with our latest AI chat services integrating community-driven smarts and flair.  

---

## Careers: Join the Hug Life!  

Looking to ride the AI wave? Hugging Face offers:

- Roles in engineering, research, community engagement, and enterprise solutions.
- An environment where your ideas turn into models shared by millions worldwide.
- A culture that's as fun as it is innovative — think open discussions, hackathons, and zero-agent humor.  
- Work flexibly while hugging the future of AI with one of the most dynamic, inclusive teams out there.  

---

## Why Hugging Face?  

Because where else can you find:

- **2 million+ ready-to-rock AI models** at your fingertips?  
- A global, buzzing community that makes AI feel like a team sport.  
- Tools and spaces so effortless, you’ll wonder why you ever wrote a model in isolation.  
- An ecosystem that’s part social club, part lab, part AI playground—and all awesome.

---

## Come Hug the Future with Us!

Visit us at [huggingface.co](https://huggingface.co) and join the revolution where code hugs creativity, and AI meets community.

---

**Hugging Face: Because AI deserves a warm embrace.** 🤗

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>